So this is where i'm gonna start working on the educational version. I'd probably want a background on both peft and bayesian techniques here

In [16]:
pip install datasets transformers peft evaluate torchmetrics scikit-learn, ipdb

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'scikit-learn,': Expected end or semicolon (after name and no valid version specifier)
    scikit-learn,
                ^


In [17]:
from datasets import load_dataset

#Download the Rotten Tomatoes dataset directly from hugging face
raw_dataset = load_dataset("rotten_tomatoes")

#shrink it
#The original training set has 8,500 reviews. We will shuffle them and grab exactly 1,000.
#We will also grab 200 for your testing/evaluation set.
#1000 doesn't seem like a lot, but for deep learning on an unimpressive cpu, this is already a lot.
small_train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = raw_dataset["test"].shuffle(seed=42).select(range(200))

print("\n--- Dataset Ready! ---") #yay!
print(f"Training examples: {len(small_train_dataset)}")
print(f"Testing examples: {len(small_test_dataset)}\n")

#A look at the very first example to see what we are working with
print(f"Text: '{small_train_dataset[0]['text']}'")
print(f"Label: {small_train_dataset[0]['label']} (0 = Negative, 1 = Positive)")


--- Dataset Ready! ---
Training examples: 1000
Testing examples: 200

Text: '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .'
Label: 0 (0 = Negative, 1 = Positive)


In [18]:
import torch
#import ipdb
from modelwrappers.wrapperbase import WrapperBase

class EducationalBayesianWrapper(WrapperBase):
    def __init__(self, model, peft_config, args, accelerator, adapter_name="default"):
        #Initialize the professor's base class (handles the optimizer, metrics, etc.)
        super().__init__(model, peft_config, args, accelerator, adapter_name)

    def forward_logits(self, batch, sample=False, n_samples=1):
        """
        The Educational Engine:
        Takes a batch of text, passes it through RoBERTa, and returns the logits.
        """
        #Extract the text tokens and attention masks from the trimmed review batch
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']

        #standard pass through the model without sampling (just one guess)
        if not sample or n_samples == 1:
            outputs = self.base_model(
                input_ids=input_ids, 
                attention_mask=attention_mask
            )
            #The professor's evaluation math expects a 3D tensor: [batch_size, n_samples, classes]
            #So we add a dimension in the middle with unsqueeze(1)
            return outputs.logits.unsqueeze(1)

        #Bayesian pass
        else:
            #force the model into training mode to activate the random dropout layers
            self.base_model.train() 
            
            stacked_logits = []
            
            #Run the exact same text through the model n_samples times (e.g., 10 times)
            for _ in range(n_samples):
                outputs = self.base_model(
                    input_ids=input_ids, 
                    attention_mask=attention_mask
                )
                stacked_logits.append(outputs.logits)
            
            #safely return the model to evaluation mode
            self.base_model.eval()

            #stack all 10 guesses together into a single block of math
            #Final Shape: [batch_size, 10, 2]
            return torch.stack(stacked_logits, dim=1)

now we wanna tokenize the data twin!!!!

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from accelerate import Accelerator


#tokenize the data. English into matrix 
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenizing(examples):
    #remember that our goal is to be able to run this on a normal computer, which is why max_length is set to 128 (instead of 512 for the full roberta)
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

#tokenize the training and testing sets
tokenized_train = small_train_dataset.map(tokenizing, batched=True)
tokenized_test = small_test_dataset.map(tokenizing, batched=True)

#then convert to pytorch tensors for the model
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

#then group the data into small manageable batches for training and evaluation
train_loader = DataLoader(tokenized_train, batch_size=16, shuffle=True)
test_loader = DataLoader(tokenized_test, batch_size=16)

#now we wanna build the base model
#load the model first
#only two labels, positive and negative
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

#parameter efficient adapter configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules = ["query", "value"],
    lora_dropout=0.1
)

#put adapter on base model
peft_model = get_peft_model(base_model, lora_config)


#initializing OUR wrapper
#prof's base class expects a huge args file. We're running ts on performance mode at 12 fps
#aka lowest settings. Note we're doing this because we are in a jupyter notebook
class NotebookArgs:
    batch_size = 16
    n_epochs = 1
    max_train_steps = 0
    outdim = 2
    opt = "adamw"
    lr = 1e-4
    opt_wd = 0.01
    adam_epsilon = 1e-8
    warmup_ratio = 0.1
    dataset_type = "bertds" # The "secret backdoor" we found earlier!
    epoch = 0
    eval_per_steps = 1000
    num_samples = 1000

accelerator = Accelerator() # Automatically routes math to your CPU

bayesian_model = EducationalBayesianWrapper(
    model=peft_model,
    peft_config=lora_config,
    args=NotebookArgs(),
    accelerator=accelerator
)

#trying it
print("\nFiring up the Bayesian Engine...")
# Grab exactly one batch of 16 reviews
test_batch = next(iter(train_loader)) 

# Push it through the Bayesian pass you just coded (asking for 5 samples)
bayesian_logits = bayesian_model.forward_logits(test_batch, sample=True, n_samples=5)

print("--- Engine Test Complete ---")
print(f"Success! Output tensor shape is: {bayesian_logits.shape}")
print("(It should read: [16, 5, 2] -> 16 reviews, 5 guesses each, 2 possible labels)")


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Firing up the Bayesian Engine...
--- Engine Test Complete ---
Success! Output tensor shape is: torch.Size([16, 5, 2])
(It should read: [16, 5, 2] -> 16 reviews, 5 guesses each, 2 possible labels)


In [20]:
import torch.nn as nn
from torch.optim import AdamW

# 1. The Setup
# The Optimizer is the mechanic that updates the LoRA "sticky notes"
optimizer = AdamW(bayesian_model.parameters(), lr=1e-4)

# The Loss Function calculates exactly how "wrong" the model's guesses are
loss_function = nn.CrossEntropyLoss()

epochs = 3 # We will run through the 1,000 reviews 3 times

print("🚀 Starting the Training Loop...\n")

for epoch in range(epochs):
    bayesian_model.train() # Turn on training mode
    total_loss = 0
    
    # Process the data 16 reviews at a time
    for step, batch in enumerate(train_loader):
        
        # Step A: Clear the old math from the previous batch
        optimizer.zero_grad()
        
        # Step B: The Forward Pass (Make a guess)
        # We use sample=False here because the Bayesian "committee" is only used for testing!
        logits = bayesian_model.forward_logits(batch, sample=False)
        
        # Our wrapper returns [16, 1, 2]. We squeeze the middle dimension out so it's just [16, 2]
        logits = logits.squeeze(1) 
        
        # Step C: Calculate the Loss (How wrong was the guess?)
        labels = batch['label']
        loss = loss_function(logits, labels)
        
        # Step D: Backpropagation (Calculate the corrections)
        loss.backward()
        
        # Step E: Update the LoRA weights
        optimizer.step()
        
        total_loss += loss.item()
        
        # Print a tiny progress update every 15 batches so you know it hasn't crashed
        if step % 15 == 0 and step > 0:
            print(f"  Batch {step} - Current Loss: {loss.item():.4f}")

    # End of Epoch
    avg_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch+1} Complete! Average Loss: {avg_loss:.4f}\n")

print("🎉 Training Complete! Your Bayesian model is fully charged.")

🚀 Starting the Training Loop...

  Batch 15 - Current Loss: 0.6890
  Batch 30 - Current Loss: 0.7039
  Batch 45 - Current Loss: 0.7160
  Batch 60 - Current Loss: 0.6814
✅ Epoch 1 Complete! Average Loss: 0.6963

  Batch 15 - Current Loss: 0.7081
  Batch 30 - Current Loss: 0.7230
  Batch 45 - Current Loss: 0.7029
  Batch 60 - Current Loss: 0.6913
✅ Epoch 2 Complete! Average Loss: 0.6930

  Batch 15 - Current Loss: 0.6801
  Batch 30 - Current Loss: 0.6928
  Batch 45 - Current Loss: 0.6568
  Batch 60 - Current Loss: 0.5481
✅ Epoch 3 Complete! Average Loss: 0.6703

🎉 Training Complete! Your Bayesian model is fully charged.


In [21]:
import torch
import torch.nn.functional as F

# The ultimate test: A complete keyboard smash
garbage_text = "The special effects were absolutely incredible and visually stunning, but the acting was completely horrible and the worst I have ever seen."

print(f"Testing Garbage Text: '{garbage_text}'")

# 1. Tokenize the text (translate to numbers)
inputs = tokenizer(garbage_text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

# 2. Run the Bayesian Committee (ask the model to guess 10 times)
with torch.no_grad():
    # Calling the engine you built!
    logits = bayesian_model.forward_logits(inputs, sample=True, n_samples=10)

# 3. Convert the raw logits into percentages (0 to 100%)
probabilities = F.softmax(logits, dim=-1)

# 4. Average all 10 guesses together to get the final confidence
mean_probabilities = probabilities.mean(dim=1).squeeze()

print(f"\n--- Final Bayesian Confidence ---")
print(f"Negative: {mean_probabilities[0].item() * 100:.2f}%")
print(f"Positive: {mean_probabilities[1].item() * 100:.2f}%")

Testing Garbage Text: 'The special effects were absolutely incredible and visually stunning, but the acting was completely horrible and the worst I have ever seen.'



--- Final Bayesian Confidence ---
Negative: 45.17%
Positive: 54.83%


In [22]:
peft_model.save_pretrained("bayesian_rotten_tomatoes_final")

and now for adrian's part 

In [23]:
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model

print("Loading a fresh model for the Standard Baseline...")

# 1. Get a completely blank brain (so we don't mix up your Bayesian weights)
standard_base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Slap fresh LoRA sticky notes on it using the exact same config from earlier
standard_model = get_peft_model(standard_base_model, lora_config)
standard_model = standard_model.to(accelerator.device) # Send it to your CPU

# 3. The Mechanic and the Loss Function
standard_optimizer = AdamW(standard_model.parameters(), lr=1e-4)
loss_function = nn.CrossEntropyLoss()

print("🚀 Starting Standard Training Loop...\n")

# Run it for the exact same 3 epochs to make it a fair fight
for epoch in range(3):
    standard_model.train() 
    total_loss = 0
    
    for step, batch in enumerate(train_loader):
        standard_optimizer.zero_grad()
        
        # STANDARD PASS: No wrapper, no committee. Just straight through the base model.
        outputs = standard_model(
            input_ids=batch['input_ids'].to(accelerator.device), 
            attention_mask=batch['attention_mask'].to(accelerator.device)
        )
        
        # Calculate loss directly from the raw logits
        loss = loss_function(outputs.logits, batch['label'].to(accelerator.device))
        
        loss.backward()
        standard_optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"✅ Epoch {epoch+1} Complete! Average Loss: {avg_loss:.4f}")

# Save Adrian's model so you have both!
standard_model.save_pretrained("standard_rotten_tomatoes_final")
print("\n🎉 Standard Baseline Complete and Saved.")

Loading a fresh model for the Standard Baseline...


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Starting Standard Training Loop...

✅ Epoch 1 Complete! Average Loss: 0.6924
✅ Epoch 2 Complete! Average Loss: 0.6932
✅ Epoch 3 Complete! Average Loss: 0.6593

🎉 Standard Baseline Complete and Saved.


confidence check

In [24]:
import torch
import torch.nn.functional as F

garbage_text = "The special effects were absolutely incredible and visually stunning, but the acting was completely horrible and the worst I have ever seen."
print(f"Testing Garbage Text on Standard Model: '{garbage_text}'")

# 1. Tokenize the text
inputs = tokenizer(garbage_text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

# Move the inputs to the CPU so they match the standard_model's location
inputs = {k: v.to(accelerator.device) for k, v in inputs.items()}

# 2. The Standard Pass (No committee, just one straight shot)
standard_model.eval() # Lock the weights
with torch.no_grad():
    # Notice we just call standard_model() directly!
    outputs = standard_model(**inputs)
    
# Extract the raw numbers
standard_logits = outputs.logits

# 3. Convert to percentages
standard_probabilities = F.softmax(standard_logits, dim=-1).squeeze()

print(f"\n--- Final Standard Confidence ---")
print(f"Negative: {standard_probabilities[0].item() * 100:.2f}%")
print(f"Positive: {standard_probabilities[1].item() * 100:.2f}%")

Testing Garbage Text on Standard Model: 'The special effects were absolutely incredible and visually stunning, but the acting was completely horrible and the worst I have ever seen.'

--- Final Standard Confidence ---
Negative: 66.33%
Positive: 33.67%
